# Import libraries and dataset into environment

In [1]:
import dill
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import lightgbm as lgb
import math as mt
import shap
from sklearn.metrics import confusion_matrix, roc_curve, auc
from MLstatkit import Bootstrapping
import gc
from joblib import Parallel, delayed, externals
import multiprocessing
num_cores = multiprocessing.cpu_count() - 1
import defined_functions
from defined_functions import sum_metric, met_collate_func
defined_functions.pd = pd

In [2]:
path = os.getcwd()
sys.path.append(path)

save_file = os.path.join(path, "session.pkl")

with open(save_file, "rb") as f:
    state = dill.load(f)

split_list_valid_smote = state["split_list_valid_smote"]
split_list_valid_smote_final = state["split_list_valid_smote_final"]
split_list_fil_valid_smote_final = state["split_list_fil_valid_smote_final"]
split_list_sim_onset_valid_smote_final = state["split_list_sim_onset_valid_smote_final"]
split_list_vldiag_valid_smote_final = state["split_list_vldiag_valid_smote_final"]
split_list_diag1_valid_smote_final = state["split_list_diag1_valid_smote_final"]
split_list_diag2_valid_smote_final = state["split_list_diag2_valid_smote_final"]

# Define functions

In [3]:
# Define function to train LightGBM model
def run_single_lgb_split(i, split):
    """Processes a single cross-validation data split loop iteration."""
    train_x = split['train'].drop(columns = ['outcome'])
    valid_x = split['valid'].drop(columns = ['outcome'])
    test_x = split['test'].drop(columns = ['outcome'])

    features = list(train_x.columns)
    cat_cols = ['age', 'gender', 'vaccination', 'comorbidity']
    
    mapping = {"Non severe": 0, "Severe": 1}
    
    train_y = split['train']['outcome'].map(mapping).astype(int)
    valid_y = split['valid']['outcome'].map(mapping).astype(int)
    test_y = split['test']['outcome'].map(mapping).astype(int)

    dtrain = lgb.Dataset(train_x, label = train_y, feature_name = features, categorical_feature = cat_cols)
    dvalid = lgb.Dataset(valid_x, label = valid_y, feature_name = features, categorical_feature = cat_cols)
    
    param = {
        'metric': ['average_precision'],
        'objective': 'binary',
        'learning_rate': 0.01,   # Shrinkage rate
        'scale_pos_weight': 1.25, # Weight of labels with positive class
        'num_leaves': 2,    # Maximum number of leaves in one tree
        'max_bin': 255, # Maximum number of bins that feature values will be bucketed in model
        'lambda_l1': 40,    # L1 regularization
        'lambda_l2': 60, # L2 regularization
        'min_gain_to_split': 3, # Minimal gain to split
        'max_delta_step': 1,  # Maximum output of tree leaves
        'bagging_fraction': 0.9,    # Proportion of data being selected for bagging without resampling
        'feature_fraction': 0.5,    # Subset features to be selected on each tree node
        'max_depth': 1,    # Maximum depth for tree model
        'min_sum_hessian_in_leaf': 30,  # Minimum sum hessian in one leaf
        'min_data_in_leaf': 120, # Minimal number of data in one leaf
        'verbosity': 0,
        'path_smooth': 40,   # Controls smoothing applied to tree nodes
        'bagging_freq': 1,    # Frequency for bagging
        'bagging_strategy': 'bagging',  # Randomly bagging sampling
        'boosting': 'gbdt', # Boosting strategy; Setting to traditional Gradient Boosting Decision Tree by default
        #'device_type': 'cuda'    # Device for the tree learning. Uncomment if running on GPU and Linux.
        'n_jobs': 1 # Setting the model to run on one thread only
    }

    lgb_model = lgb.train(
        train_set = dtrain,
        num_boost_round =  5000,  # Number of boosting iterations
        params = param,
        valid_sets = dvalid, # Setting validation set
        valid_names = ["valid"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds = 300,
                first_metric_only = True
                ),
            lgb.log_evaluation(100)
            ],
        keep_training_booster = True
    )
    del dtrain, dvalid
    gc.collect()

    # Make predictions on Testing Set
    lgb_pred_prob = lgb_model.predict(test_x)
    lgb_pred = (lgb_pred_prob >= 0.5).astype(int)
    lgb_pred_res = np.where(lgb_pred == 0, "Non severe", "Severe")
    
    # Calculate Area Under the ROC curve
    fpr, tpr_curve, _ = roc_curve(test_y, lgb_pred_prob, pos_label = 1)
    auc_val_lgb = auc(fpr, tpr_curve)
    
    # Store standard structure dictionary to act like R's pROC container
    roc_container = {
        'actual': test_y,
        'probabilities': lgb_pred_prob
    }
    
    # Calculate Area Under the Precision-Recall Curve (AUPRC / PR-AUC)
    auprc_val, prc_ci_lower, prc_ci_upper = Bootstrapping(test_y, lgb_pred_prob, 'pr_auc')
   
    # Compute Confusion Matrix (Test)
    tn, fp, fn, tp = confusion_matrix(test_y, lgb_pred, labels = [0, 1]).ravel()
    
    # Format a formal R-styled evaluation matrix lookup dataframe 
    cfm_lgb = pd.DataFrame(
        [[tn, fp], [fn, tp]], 
        index = ["Non severe", "Severe"], 
        columns = ["Non severe", "Severe"]
    )
    cfm_lgb.index.name = 'Prediction'
    cfm_lgb.columns.name = 'Observed'
    
    # Calculate performance metrics
    accuracy_lgb = (tp + tn) / (tn + fp + fn + tp) if (tn + fp + fn + tp) > 0 else 0
    sensitivity_lgb = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity_lgb = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv_lgb = tn / (tn + fn) if (tn + fn) > 0 else 0
    precision_lgb = tp / (tp + fp) if (tp + fp) > 0 else 0
        
    # Make predictions on Training Set to gather accuracy
    lgb_pred_train_prob = lgb_model.predict(train_x)
    lgb_pred_train = (lgb_pred_train_prob >= 0.5).astype(int)
    tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(train_y, lgb_pred_train, labels = [0, 1]).ravel()
    accuracy_lgb_train = (tp_tr + tn_tr) / (tn_tr + fp_tr + fn_tr + tp_tr)
    
    explainer = shap.TreeExplainer(model = lgb_model)
    shap_values = explainer(train_x).values.astype(np.float32)
    
    return {
        "model": lgb_model,
        "confusion_matrix": cfm_lgb,
        "accuracy_training": accuracy_lgb_train,
        "accuracy_testing": accuracy_lgb,
        "sensitivity": sensitivity_lgb,
        "specificity": specificity_lgb,
        "precision": precision_lgb,
        "npv": npv_lgb,
        "roc": roc_container,
        "AUC_value": auc_val_lgb,
        "PRC_val": auprc_val,
        "PRC_lower_ci": prc_ci_lower,
        "PRC_upper_ci": prc_ci_upper,
        "SHAP_values": shap_values,
        "prediction": lgb_pred_res,
        "pred_prob": lgb_pred_prob
    }

# Define function to train LightGBM for 100 times in parallel
def model_func_lgb_tune(data_list):
    """Main execution wrapper to distribute LightGBM tuning loops over system CPU threads."""
    # Count system resource availability profiles
    num_cores = multiprocessing.cpu_count() - 1
    
    if __name__ == '__main__':
        try:
            result_list = Parallel(n_jobs = num_cores)(
                delayed(run_single_lgb_split)(i, data_list[i]) 
                for i in range(100)
                )
        finally:
            externals.loky.get_reusable_executor().shutdown(wait = True)
            gc.collect()

    return result_list

# Fitting dataset into the model

## Fitting data list without VL information

In [4]:
lgb_fil_list = model_func_lgb_tune(split_list_fil_valid_smote_final)
lgb_fil_met_summary = sum_metric(lgb_fil_list)
lgb_fil_metrics_summary = lgb_fil_met_summary["metric_summary"]
lgb_fil_summary = met_collate_func(lgb_fil_metrics_summary).assign(
    models = "LightGBM (No VL info & SMOTE)"
)
lgb_fil_summary

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,38.1486,17.7473,62.3000,LightGBM (No VL info & SMOTE)
1,AUROC_value,91.3522,80.2948,96.9942,LightGBM (No VL info & SMOTE)
2,Accuracy,77.6828,57.5453,89.1390,LightGBM (No VL info & SMOTE)
3,Accuracy_train,83.8505,72.4312,89.8585,LightGBM (No VL info & SMOTE)
4,NPV,99.5654,98.3690,100.0000,LightGBM (No VL info & SMOTE)
5,Precision,12.5191,5.3598,20.4884,LightGBM (No VL info & SMOTE)
6,Sensitivity,89.1000,64.7500,100.0000,LightGBM (No VL info & SMOTE)
7,Specificity,77.3271,56.3863,89.2601,LightGBM (No VL info & SMOTE)


## Fitting data list with simulated VL at symptom onset

In [5]:
lgb_vlsymp_sim_list = model_func_lgb_tune(split_list_sim_onset_valid_smote_final)
lgb_vlsymp_sim_met_summary = sum_metric(lgb_vlsymp_sim_list)
lgb_vlsymp_sim_metrics_summary = lgb_vlsymp_sim_met_summary["metric_summary"]
lgb_vlsymp_sim_summary = met_collate_func(lgb_vlsymp_sim_metrics_summary).assign(
    models = "LightGBM (VL symp simulated & SMOTE)"
)
lgb_vlsymp_sim_summary

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,39.0655,18.7118,61.7642,LightGBM (VL symp simulated & SMOTE)
1,AUROC_value,91.6016,80.5093,96.6713,LightGBM (VL symp simulated & SMOTE)
2,Accuracy,80.4139,58.1495,95.0227,LightGBM (VL symp simulated & SMOTE)
3,Accuracy_train,84.0447,73.8162,90.0584,LightGBM (VL symp simulated & SMOTE)
4,NPV,99.4324,98.0921,100.0000,LightGBM (VL symp simulated & SMOTE)
5,Precision,15.9241,5.6703,33.3333,LightGBM (VL symp simulated & SMOTE)
6,Sensitivity,84.1000,50.0000,100.0000,LightGBM (VL symp simulated & SMOTE)
7,Specificity,80.2991,57.1573,96.1137,LightGBM (VL symp simulated & SMOTE)


## Fitting data list with VL at diagnosis

In [6]:
lgb_vldiag_list = model_func_lgb_tune(split_list_vldiag_valid_smote_final)
lgb_vldiag_met_summary = sum_metric(lgb_vldiag_list)
lgb_vldiag_metrics_summary = lgb_vldiag_met_summary["metric_summary"]
lgb_vldiag_summary = met_collate_func(lgb_vldiag_metrics_summary).assign(
    models = "LightGBM (VL diag & SMOTE)"
)
lgb_vldiag_summary

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,41.2322,23.9557,60.7583,LightGBM (VL diag & SMOTE)
1,AUROC_value,92.5391,79.1507,97.6402,LightGBM (VL diag & SMOTE)
2,Accuracy,85.6314,63.1269,95.0227,LightGBM (VL diag & SMOTE)
3,Accuracy_train,85.8396,77.2884,90.6893,LightGBM (VL diag & SMOTE)
4,NPV,99.3207,98.2282,100.0000,LightGBM (VL diag & SMOTE)
5,Precision,18.7926,7.0556,34.7333,LightGBM (VL diag & SMOTE)
6,Sensitivity,80.5000,50.0000,100.0000,LightGBM (VL diag & SMOTE)
7,Specificity,85.7913,62.7648,96.1137,LightGBM (VL diag & SMOTE)


## Fitting data list with VL at diagnosis & VL at 1-day after diagnosis

In [7]:
lgb_add1_list = model_func_lgb_tune(split_list_diag1_valid_smote_final)
lgb_add1_met_summary = sum_metric(lgb_add1_list)
lgb_add1_metrics_summary = lgb_add1_met_summary["metric_summary"]
lgb_add1_summary = met_collate_func(lgb_add1_metrics_summary).assign(
    models = "LightGBM (VL diag + 1 & SMOTE)"
)
lgb_add1_summary

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,42.6251,24.2361,64.3404,LightGBM (VL diag + 1 & SMOTE)
1,AUROC_value,92.5816,84.1452,97.9860,LightGBM (VL diag + 1 & SMOTE)
2,Accuracy,87.5589,64.3051,95.1662,LightGBM (VL diag + 1 & SMOTE)
3,Accuracy_train,85.6796,77.4831,90.8917,LightGBM (VL diag + 1 & SMOTE)
4,NPV,99.3083,98.3187,100.0000,LightGBM (VL diag + 1 & SMOTE)
5,Precision,20.6215,7.7700,34.9242,LightGBM (VL diag + 1 & SMOTE)
6,Sensitivity,79.7000,50.0000,100.0000,LightGBM (VL diag + 1 & SMOTE)
7,Specificity,87.8037,63.1931,96.2617,LightGBM (VL diag + 1 & SMOTE)


## Fitting data list with VL at diagnosis & VL at 2-days after diagnosis

In [8]:
lgb_add2_list = model_func_lgb_tune(split_list_diag2_valid_smote_final)
lgb_add2_met_summary = sum_metric(lgb_add2_list)
lgb_add2_metrics_summary = lgb_add2_met_summary["metric_summary"]
lgb_add2_summary = met_collate_func(lgb_add2_metrics_summary).assign(
    models = "LightGBM (VL diag + 2 & SMOTE)"
)
lgb_add2_summary

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,41.8833,20.8132,61.6987,LightGBM (VL diag + 2 & SMOTE)
1,AUROC_value,92.9051,82.8201,98.0495,LightGBM (VL diag + 2 & SMOTE)
2,Accuracy,88.4955,68.2704,95.1662,LightGBM (VL diag + 2 & SMOTE)
3,Accuracy_train,86.5493,80.8827,91.2630,LightGBM (VL diag + 2 & SMOTE)
4,NPV,99.2908,98.3735,100.0000,LightGBM (VL diag + 2 & SMOTE)
5,Precision,21.0364,8.7673,36.1909,LightGBM (VL diag + 2 & SMOTE)
6,Sensitivity,79.1000,50.0000,100.0000,LightGBM (VL diag + 2 & SMOTE)
7,Specificity,88.7882,67.2819,96.2617,LightGBM (VL diag + 2 & SMOTE)


# Save model trained

In [ ]:
path = os.getcwd()

state = {
    "lgb_fil_list": lgb_fil_list,
    "lgb_vlsymp_sim_list": lgb_vlsymp_sim_list,
    "lgb_vldiag_list": lgb_vldiag_list,
    "lgb_add1_list": lgb_add1_list,
    "lgb_add2_list": lgb_add2_list
}

save_file = os.path.join(path, "lgb_trained.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

print(f"Saved to: {save_file}")

In [ ]:
path = os.getcwd()

state = {
    "lgb_fil_summary": lgb_fil_summary,
    "lgb_vlsymp_sim_summary": lgb_vlsymp_sim_summary,
    "lgb_vldiag_summary": lgb_vldiag_summary,
    "lgb_add1_summary": lgb_add1_summary,
    "lgb_add2_summary": lgb_add2_summary
}

save_file = os.path.join(path, "lgb_metric_summary.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

print(f"Saved to: {save_file}")